In [424]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [425]:
# Material Parameters
mu = 0.8
jm = 40.0
Identity = torch.eye(3)

In [426]:
# Gent Stress Function
def gent_stress_function(lamda, mu, jm):
    I1 = lamda**2 + 2/lamda
    stress = mu * (lamda - 1/lamda**2) / (1 - (I1 - 3)/jm)
    return stress

In [427]:
def deformation_gradient(lamda):
    diagonal_values = torch.stack([
        lamda,
        lamda**(-0.5),
        lamda**(-0.5)
    ], dim=-1)

    return torch.diag_embed(diagonal_values)

In [428]:
def frobenius_norm(F):
    return torch.sqrt(
        torch.sum((F - Identity)**2, dim=(-2, -1))
    )

In [429]:
def add_noise(P_true_train,eta_min,eta_max,lamda):
    P_char = torch.max(torch.abs(P_true_train))
    noise_sd_min = eta_min * P_char
    noise_sd_max = eta_max * P_char
    print("noise_sd_min:", noise_sd_min.item())
    print("noise_sd_max:", noise_sd_max.item()) 
    F = deformation_gradient(lamda)
    F_lambdamax = deformation_gradient(torch.max(lamda))
    q = 2.0
    t = frobenius_norm(F)
    ksy = torch.randn_like(P_true_train)
    conditional_noise_variance = torch.square(noise_sd_min) + (torch.square(noise_sd_max) - torch.square(noise_sd_min)) * ((t / frobenius_norm(F_lambdamax))**q)
    return torch.sqrt(conditional_noise_variance) * ksy

In [430]:
# Synthetic Data Generation
def generate_synthetic_data(num_samples, lamda_min, lamda_max,eta_min,eta_max):
    lamda_train = torch.linspace(lamda_min, lamda_max, num_samples)
    I1_train = lamda_train**2 + 2/lamda_train
    P_true_train = gent_stress_function(lamda_train, mu, jm)   
    noise = add_noise(P_true_train,eta_min,eta_max,lamda_train)
    Y_train = P_true_train + noise
    Y_train = Y_train.reshape(-1, 1)
    lamda_train =lamda_train.reshape(-1, 1)
    I1_train = I1_train.reshape(-1, 1)
    x = torch.cat((lamda_train, I1_train,Y_train), dim=1)
    return x


In [431]:
x = generate_synthetic_data(num_samples=50000, lamda_min=1.0, lamda_max=4.0, eta_min=0.005,eta_max=0.03)

noise_sd_min: 0.02377358451485634
noise_sd_max: 0.14264149963855743


In [432]:
class NeuralNetwork(nn.Module):
    def __init__(self,m = 10):
        super().__init__()
        self.a = nn.Parameter(torch.randn(1))
        self.c = nn.Parameter(torch.randn(m))
        self.w = nn.Parameter(torch.randn(m))
        self.b = nn.Parameter(torch.randn(m))
    
    def forward(self,I1):
        a = F.softplus(self.a)
        c = F.softplus(self.c)
        w = F.softplus(self.w)
        x = I1 - 3
        z = x * w + self.b

        # Broadcasting (element wise multiplication)

        hidden = (F.softplus(z) - F.softplus(self.b))
        psi = (a * x + torch.sum(c * hidden, dim = -1, keepdim = True))
        # sum across the last dimension to get the final output (sum across the columns)
        return psi

In [433]:
def weighted_loss(model,data,noise_variance):
    lamda = data[:,0].reshape(-1,1)
    I1 = data[:,1].reshape(-1,1)
    P_true = data[:,2].reshape(-1,1)
    I1.requires_grad_(True)

    P_pred = model(I1)
    # Compute the gradient of the predicted energy potential with respect to I1
    dW_dI1 = torch.autograd.grad(P_pred, I1, grad_outputs=torch.ones_like(P_pred), create_graph=True)[0]
    P_pred = 2 * dW_dI1 * (lamda - 1/lamda**2)

    squared_error= (P_pred - P_true)**2
    if noise_variance is None:
        loss = torch.mean(squared_error)
    else:
        loss = torch.mean(squared_error / noise_variance)

    return loss,P_pred

In [434]:
model = NeuralNetwork(m=32)

In [435]:
def Backpropagation(model, data, noise_variance, learning_rate=0.01,num_epochs=1000):
    model = NeuralNetwork()
    

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
      optimizer.zero_grad()
      loss,P_pred = weighted_loss(model, data, noise_variance)
    
      loss.backward()
      optimizer.step()
    
      if epoch % 100 == 0:
          print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.10f}')  

In [436]:
Backpropagation(model, x, noise_variance=100, learning_rate=0.01,num_epochs=1)

Epoch [0/1], Loss: 4.4690198898
